# Data Understanding

In [1]:
import pandas as pd
import numpy as np

In [2]:
#dataset loading
data=pd.read_csv(r"C:\Users\vines\OneDrive\Desktop\data\twitter_training.csv")

In [3]:
#changing names to meaningful names
data.columns=['Id','Entity','Sentiment','Tweet']

In [4]:
#dataset info
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         74681 non-null  int64 
 1   Entity     74681 non-null  object
 2   Sentiment  74681 non-null  object
 3   Tweet      73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [21]:
# droping null
data.dropna(subset=['Tweet'], inplace=True)

#  Remove 'Irrelevant' sentiment
data=data[data['Sentiment'] != 'Irrelevant']

In [5]:
#first five rows
data.head()

,Id,Entity,Sentiment,Tweet
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [6]:
#columns of the dataset
print(data.columns)

Index(['Id', 'Entity', 'Sentiment', 'Tweet'], dtype='object')


In [7]:
#Total number of samples 
print("Total Samples")
data.shape

Total Samples


(74681, 4)

In [8]:
#class distribution
print("Class Distribution")
data['Sentiment'].value_counts()

Class Distribution


Sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [9]:
#sample Texts
print("Sample Texts")
data['Tweet'][2]

Sample Texts


'im coming on borderlands and i will murder you all,'

In [24]:
#label encoder for coverting string to numeric
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
data['Sentiment_Label'] = le.fit_transform(data['Sentiment'])
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}


# NLP Preprocessing

In [10]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words=set(stopwords.words("english"))
lemmatizer=WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vines\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vines\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [11]:
def preprocess_text(text):
    if pd.isnull(text):
        return ""
    #lowercasing
    text=str(text).lower()

    #removing punctuation
    text=text.translate(str.maketrans("","",string.punctuation))

    #removing numbers
    text=re.sub(r'/d',"",text)

    #removing urls and mentions
    text=re.sub(r'https?\S+|www\S+|@\W+','',text)

    #removing hashtags
    text=re.sub(r'#',"",text)

    #tokenization
    words=text.split()

    #removing stop words and lemmatization
    words=[lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)
    


In [12]:
data['clean_text']=data['Tweet'].apply(preprocess_text)

data[['Tweet','clean_text']].head()

,Tweet,clean_text
0,I am coming to the borders and I will kill you...,coming border kill
1,im getting on borderlands and i will kill you ...,im getting borderland kill
2,im coming on borderlands and i will murder you...,im coming borderland murder
3,im getting on borderlands 2 and i will murder ...,im getting borderland 2 murder
4,im getting into borderlands and i can murder y...,im getting borderland murder


# Feature Engineering

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(max_features=5000,stopwords="english")

x=tfidf.fit_transform(data['clean_text'])

y=data['Sentiment']

# Model Building

In [14]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [15]:
#LOGISTIC REGRESSION
from sklearn.linear_model import LogisticRegression

lr=LogisticRegression(max_iter=200)
lr.fit(x_train,y_train)
y_pred_lr=lr.predict(x_test)

In [16]:
# Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb=MultinomialNB()
nb.fit(x_train,y_train)
y_pred_nb=nb.predict(x_test)

In [17]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

dt=DecisionTreeClassifier()
dt.fit(x_train,y_train)
y_pred_dt=dt.predict(x_test)

# Model Evalution

In [29]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

def evalute(y_test,y_pred,name):
    print(f"\n{name} Evalution")
    print("Accuracy:",accuracy_score(y_test,y_pred))
    print("Precision:",precision_score(y_test,y_pred,average='weighted'))
    print("Recall:",recall_score(y_test,y_pred,average='weighted'))
    print("F1 Score:",f1_score(y_test,y_pred,average='weighted'))

In [30]:
evalute(y_test,y_pred_lr,"Logistic Regression")
evalute(y_test,y_pred_nb,"Naive Bayes")
evalute(y_test,y_pred_dt,"Decision Tree")


Logistic Regression Evalution
Accuracy: 0.639485840530227
Precision: 0.6387558596919236
Recall: 0.639485840530227
F1 Score: 0.636621346628586

Naive Bayes Evalution
Accuracy: 0.6028653678784227
Precision: 0.6088556783794818
Recall: 0.6028653678784227
F1 Score: 0.5874009101550177

Decision Tree Evalution
Accuracy: 0.7699002477070362
Precision: 0.7728790183477885
Recall: 0.7699002477070362
F1 Score: 0.769705665963552


# Comparison

In [31]:
result=pd.DataFrame({
    "Model":["Logistic Regression","Naive Bayes","Decision Tree"],
    "Acuuracy":[accuracy_score(y_test,y_pred_lr),accuracy_score(y_test,y_pred_nb),accuracy_score(y_test,y_pred_dt)]
})
print(result)

                 Model  Acuuracy
0  Logistic Regression  0.639486
1          Naive Bayes  0.602865
2        Decision Tree  0.769900


# Insights

Clearly Decision Tree is the winner

1. Why Decision Trees Won (Non-Linearity)
Sentiment in tweets is often complex and non-linear. A word like "Great" might be positive, but "Not great" is negative. Decision Trees are excellent at capturing these specific combinations of words (feature interactions) that Logistic Regression might miss. The 77% accuracy suggests it is picking up on specific "branches" of vocabulary that define your sentiments well.

2. Logistic Regression's "Linear" Struggle
A 64% score is decent but indicates that the relationship between your TF-IDF words and the sentiment isn't perfectly linear. It is likely struggling with sarcasm or tweets where positive and negative words are mixed together.

3. Naive Bayes and the "Independence" Trap
Naive Bayes assumes that every word in a tweet is independent of the others. In 74,000 tweets, this is rarely true. It often struggles with "Neutral" sentiments because they don't have strong, independent "trigger" words like "amazing" or "terrible."